In [ ]:
import asyncio
import gc
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import requests
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

sys.path.append("src")
from run_bench import run_benchmark

In [2]:
BASE_MODELS_DIR = Path("models")
BASE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

VLLM_HOST = "127.0.0.1"
VLLM_PORT = 8000
VLLM_API = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
VLLM_KEY = "local-bench-key"

HF_TOKEN = os.getenv("HF_TOKEN", None)

BENCH_CONCURRENCY = 40

VLLM_GPU_MEMORY_UTIL = 0.90
VLLM_MAX_NUM_SEQS = 20
VLLM_MAX_MODEL_LEN = 8192

# МОДЕЛЬ ДОЛЖНА ЛЕЖАТЬ В ПАПКЕ BASE_MODELS_DIR/alias-модели
MODEL_SPECS = [
    ######{"alias": "Qwen3-8B", "repo_id": "Qwen/Qwen3-8B"},
    ######{"alias": "YandexGPT-5-Lite-8B-instruct", "repo_id": "yandex/YandexGPT-5-Lite-8B-instruct"},
    ######{"alias": "RuadaptQwen3-8B-Hybrid", "repo_id": "RefalMachine/RuadaptQwen3-8B-Hybrid"},
    
    # --- ВСЕ ЧТО ВЫШЕ - ПОСЧИТАНО ---
    
    #{"alias": "T-pro-it-2.1", "repo_id": "t-tech/T-pro-it-2.1"},
    #{"alias": "T-lite-it-2.1", "repo_id": "t-tech/T-lite-it-2.1"},
    #{"alias": "T-pro-it-2.0", "repo_id": "t-tech/T-pro-it-2.0"},
    #{"alias": "Qwen3-32B", "repo_id": "Qwen/Qwen3-32B"},
    #{"alias": "RuadaptQwen3-32B-Instruct", "repo_id": "RefalMachine/RuadaptQwen3-32B-Instruct"},
    #{"alias": "Qwen3-30B-A3B-Instruct-2507", "repo_id": "Qwen/Qwen3-30B-A3B-Instruct-2507"},
    #{"alias": "Qwen3-14B", "repo_id": "Qwen/Qwen3-14B"},
    
    #{"alias": "Qwen3-4B-Instruct", "repo_id": "Qwen/Qwen3-4B-Instruct-2507"},
    #{"alias": "RuadaptQwen3-4B-Instruct", "repo_id": "RefalMachine/RuadaptQwen3-4B-Instruct"},
    #{"alias": "Qwen3-4B", "repo_id": "Qwen/Qwen3-4B"},
    
    #{"alias": "avibe", "repo_id": "AvitoTech/avibe"},
    #{"alias": "GigaChat3-10B-A1.8B-bf16", "repo_id": "ai-sage/GigaChat3-10B-A1.8B-bf16"},
    #{"alias": "T-lite-it-1.0", "repo_id": "t-tech/T-lite-it-1.0"},
    #{"alias": "Vikhr-Nemo-12B-Instruct-R-21-09-24", "repo_id": "Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24"},
    #{"alias": "RuadaptQwen2.5-7B-Lite-Beta", "repo_id": "RefalMachine/RuadaptQwen2.5-7B-Lite-Beta"},
]

In [3]:
# ДЛЯ СКАЧИВАНИЯ МОДЕЛЕЙ

# def local_dir_for(alias: str) -> Path:
#     safe = alias.replace("/", "_").replace(" ", "_")
#     return BASE_MODELS_DIR / safe
# 
# 
# def download_all_models():
#     for spec in MODEL_SPECS:
#         local_dir = local_dir_for(spec["alias"])
#         spec["local_dir"] = str(local_dir)
# 
#         if local_dir.exists() and any(local_dir.iterdir()):
#             print(f"[SKIP] Уже скачана: {spec['alias']} -> {local_dir}")
#             continue
# 
#         print(f"[DOWNLOAD] {spec['alias']} <- {spec['repo_id']}")
#         snapshot_download(
#             repo_id=spec["repo_id"],
#             local_dir=str(local_dir),
#             token=HF_TOKEN,
#         )
#         print(f"[OK] {spec['alias']} -> {local_dir}")
# 
# download_all_models()

In [3]:
def wait_vllm_ready(timeout_sec: int = 1200):
    url = f"{VLLM_API}/models"
    headers = {"Authorization": f"Bearer {VLLM_KEY}"}
    deadline = time.time() + timeout_sec
    last_error = None

    while time.time() < deadline:
        try:
            r = requests.get(url, headers=headers, timeout=5)
            if r.ok:
                return
            last_error = f"{r.status_code}: {r.text[:200]}"
        except Exception as e:
            last_error = repr(e)

        time.sleep(2)

    raise RuntimeError(f"vLLM не поднялся. Последняя ошибка: {last_error}")


def start_vllm_server(spec: dict) -> subprocess.Popen:
    alias = spec["alias"]
    model_path = BASE_MODELS_DIR / alias.replace("/", "_").replace(" ", "_")

    cmd = [
        "vllm", "serve", model_path,
        "--host", VLLM_HOST,
        "--port", str(VLLM_PORT),
        "--api-key", VLLM_KEY,
        "--served-model-name", alias,
        "--tensor-parallel-size", "1",
        "--dtype", "auto",
        "--gpu-memory-utilization", str(VLLM_GPU_MEMORY_UTIL),
        "--max-model-len", str(VLLM_MAX_MODEL_LEN),
        "--max-num-seqs", str(VLLM_MAX_NUM_SEQS),
        "--swap-space", "16",
        "--generation-config", "vllm",
        "--disable-log-stats",
        "--disable-uvicorn-access-log",
        "--disable-log-requests"
    ]

    # cmd.append("--trust-remote-code")

    print(f"\n=== START vLLM: {alias} ===")
    env = os.environ.copy()
    env["VLLM_CONFIGURE_LOGGING"] = "0"
    
    proc = subprocess.Popen(
        cmd,
        start_new_session=True,
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    wait_vllm_ready()
    print(f"[READY] {alias}")
    return proc


def stop_vllm_server(proc: subprocess.Popen):
    print("[STOP] vLLM")

    if proc and proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass

        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait(timeout=10)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    print("[CLEARED] CUDA cache")

In [4]:
async def run_benchmarks_for_model(model_name: str, encoder, encoder_device):
    common_kwargs = dict(
        api=VLLM_API,
        key=VLLM_KEY,
        model_name=model_name,
        concurrency=BENCH_CONCURRENCY,
        number_of_books=40,
        encoder_name="deepvk/USER-bge-m3",
        device="cpu",
        initial_word_limit=500,
        cap_chars=80000,
        shared_encoder=encoder,
        shared_device=encoder_device,
    )

    runs = [
        ("output_hierarchical_default", "hierarchical", "default"),
        ("output_hierarchical_filtered", "hierarchical", "filtered"),
        ("output_blueprint_default", "blueprint", "default"),
        ("output_blueprint_cluster", "blueprint", "cluster"),
    ]

    for output_dir, method, mode in runs:
        print(f"  -> {model_name}: {method}/{mode}")
        start = time.perf_counter()
        await run_benchmark(
            output_dir=output_dir,
            method=method,
            mode=mode,
            **common_kwargs,
        )
        end = time.perf_counter()
        print(f"Time for  -> {model_name}: {method}/{mode}: {end - start}")

In [5]:
# =========================
# ГЛАВНЫЙ ЦИКЛ
# =========================

async def main():

    encoder_device = torch.device("cuda")
    encoder = SentenceTransformer("deepvk/USER-bge-m3").to(encoder_device)

    # 3) vLLM -> bench -> no vLLM -> clear memo
    for spec in MODEL_SPECS:
        proc = None
        try:
            print(f"\n########## {spec['alias']} ##########")
            proc = start_vllm_server(spec)
            await run_benchmarks_for_model(spec["alias"], encoder, encoder_device)
        except Exception as e:
            print(f"[ERROR] {spec['alias']}: {e}")
        finally:
            if proc is not None:
                stop_vllm_server(proc)

    print("\nГотово.")

In [ ]:
await main()